# CLOY Processing

This notebook is for you to work with the CLOY data folder.

In [1]:
from textblob import TextBlob
import json

# Table of Contents

- [Step 1: Turn every thread into a vector](#sec1)
- [Step 2: Calculate the similarity](#sec2)
- [Step 3: Write function getSimilar()](#sec3)
- [Step 4: Verification through thread_explorer](#sec4)
- [Step 5: Evaluate the Influence of Stop Words](#sec5)

<a id="sec1"></a>
## Step 1: Turn every thread into a vector

In [3]:
loaded_cloy = []
with open("cloy_chronological.jsonl", "r", encoding="utf-8") as f: 
    for line in f: 
        loaded_cloy.append(json.loads(line))

In [5]:
# check the nested data structure 
loaded_cloy[:5]

[{'post_id': 'dkxewd',
  'timestamp': 1571646432,
  'author': '[deleted]',
  'subreddit': 'CrashLandingOnYou',
  'title': 'Crash Landing On You has been created',
  'selftext': '[deleted]',
  'url': 'https://www.reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/',
  'score': 1,
  'permalink': 'https://reddit.com/r/CrashLandingOnYou/comments/dkxewd/crash_landing_on_you_has_been_created/',
  'comments': []},
 {'post_id': 'ffh52z',
  'timestamp': 1583693959,
  'author': 'bruisedlee123',
  'subreddit': 'CrashLandingOnYou',
  'title': 'ok, let’s get this started! What are your thoughts about the show?',
  'selftext': 'I haven’t seen a kdrama for the longest time and it was actually my dad who chose to stream CLOY on Netflix. He stopped watching after the second episode but I went on after immediately falling in love with the leads. Hyun Bin is just so charismatic as Capt. Ri Jeong Hyeok and Son Ye-jin is perfect as Yoon Se-ri. She’s definitely got the rom-

In [6]:
wordlist = []

for thread in loaded_cloy:
    # start with the title and the post text
    alltext = thread["title"] + " " + thread["selftext"]
    # comments still to read (we start with the top-level ones)
    readin = list(thread["comments"])
    while len(readin) > 0:
        c = readin.pop()                  # take one comment off the list
        alltext = alltext + " " + c["comment_body"]
        # if this comment has replies, add them to the list so we read them too
        if "replies" in c:
            readin.extend(c["replies"])
    # one entry per thread
    wordlist.append(TextBlob(alltext).words)

In [7]:
# check the results
len(wordlist)

3075

In [8]:
import pandas as pd
import numpy as np
from collections import Counter

# every unique word becomes one dimension
vocab = sorted(set(w for words in wordlist for w in words))
df = pd.DataFrame(
    [Counter(words) for words in wordlist]
).reindex(columns=vocab).fillna(0).astype(int)

df.index = [f"d{i+1}" for i in range(len(df))]

In [11]:
df.head

<bound method NDFrame.head of        '-i  '-sshi  'D  'LL  'M  'RE  'S  'd  'll  'm  ...  🫰🫰🫰🫰  🫶  🫶🍱  🫶🏻  \
d1       0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d2       0       0   0    0   0    0   0   1    2   7  ...     0  0   0   0   
d3       0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d4       0       0   0    0   0    0   0   1    0   1  ...     0  0   0   0   
d5       0       0   0    0   0    0   0   2    1   3  ...     0  0   0   0   
...    ...     ...  ..  ...  ..  ...  ..  ..  ...  ..  ...   ... ..  ..  ..   
d3071    0       0   0    0   0    0   0   0    1   0  ...     0  0   0   0   
d3072    0       0   0    0   0    0   0   0    0   1  ...     0  0   0   0   
d3073    0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d3074    0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d3075    0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   

       🫶🏻😂😂  🫶🏻🫶🏻🫶🏻  

<a id="sec2"></a>
## Step 2: Calculate the similarity

In [12]:
# calculate cosine similarity
from numpy.linalg import norm
import math
from itertools import combinations

def cosineSimilarity(vec1, vec2): 
    v1=np.array(vec1)
    v2=np.array(vec2)
    denom = norm(v1)*norm(v2)
    return np.dot(v1, v2)/denom if denom else 0

In [13]:
# turn the dataframe into a list of vectors to save computational expense
M = df.values

scores = []
for i, j in combinations(range(len(df)), 2):
    score = cosineSimilarity(M[i], M[j])
    scores.append({
        "thread 1": loaded_cloy[i]["post_id"],
        "thread 2": loaded_cloy[j]["post_id"],
        "similarity score": score
    })

In [15]:
scores[:5]

[{'thread 1': 'dkxewd',
  'thread 2': 'ffh52z',
  'similarity score': np.float64(0.007144315315199139)},
 {'thread 1': 'dkxewd',
  'thread 2': 'fguorr',
  'similarity score': np.float64(0.0)},
 {'thread 1': 'dkxewd',
  'thread 2': 'fhcrpl',
  'similarity score': np.float64(0.0)},
 {'thread 1': 'dkxewd',
  'thread 2': 'fj5byj',
  'similarity score': np.float64(0.011499064780761866)},
 {'thread 1': 'dkxewd',
  'thread 2': 'fjimic',
  'similarity score': np.float64(0.0)}]

In [16]:
scores_df = pd.DataFrame(scores)
scores_df.to_csv('cos_similarity.csv')

<a id="sec3"></a>
## Step 3: Write function getSimilar()

In [17]:
titles = {thread["post_id"]: thread["title"] for thread in loaded_cloy}
scores_df_sorted = scores_df.sort_values(by="similarity score", ascending=False)

def getSimilar(threadID):
    sim = scores_df_sorted[scores_df_sorted["thread 1"] == threadID][:5]
    title = titles[threadID]
    top5 = []
    for index, row in sim.iterrows():
        top5.append(titles[row["thread 2"]])
    print("Thread ID: ", threadID)
    print("Thread Title: ", title)
    print("Top 5 Subject Titles: ", top5)

In [18]:
# Test getSimilar()
getSimilar("1hm20kg")

Thread ID:  1hm20kg
Thread Title:  This scene really hits me hard.
Top 5 Subject Titles:  ['This scene really hits me hard.', "Both hurting, both won't let go. ", 'Underrated moment #7: "Se-ri you just woke up. You can\'t take off his shirt like that already!"', "It's Hyun Bin's birthday! Share your favourite moments of HB from CLOY! Gifs especially welcome 🤗", "I've watched this show over a dozen times but Seo Dan and Gu Seung Jun really grew on me"]


<a id="sec4"></a>
## Step 4: Verification through thread_explorer

I searched the original thread and the five most similar threads in thread_explorer. 
The results appeared reasonable because several of the retrieved threads discussed the same characters and plot events as the original thread. 
Some threads used different wording but were still semantically related. 

<a id="sec5"></a>
## Step 5: Evaluate the Influence of Stop Words

In [19]:
# check stop words
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

stopWords = stopwords.words("english")

from nltk.corpus import stopwords

stopWords = stopwords.words("english")
print(stopWords)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/zhaoyiran/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [20]:
# remove stop words from the threads
newWordlist = []

for wl in wordlist:
    newWl = []
    for w in wl:
        if w.lower() not in stopWords:
            newWl.append(w)
    newWordlist.append(newWl)

In [21]:
newWordlist[:5]

[['Crash', 'Landing', 'created', 'deleted'],
 ['ok',
  'let',
  '’',
  'get',
  'started',
  'thoughts',
  'show',
  '’',
  'seen',
  'kdrama',
  'longest',
  'time',
  'actually',
  'dad',
  'chose',
  'stream',
  'CLOY',
  'Netflix',
  'stopped',
  'watching',
  'second',
  'episode',
  'went',
  'immediately',
  'falling',
  'love',
  'leads',
  'Hyun',
  'Bin',
  'charismatic',
  'Capt',
  'Ri',
  'Jeong',
  'Hyeok',
  'Son',
  'Ye-jin',
  'perfect',
  'Yoon',
  'Se-ri',
  '’',
  'definitely',
  'got',
  'rom-com',
  'drama',
  'genre',
  'pat',
  'also',
  'loved',
  'Capt',
  'Ri',
  '’',
  'squad',
  'especially',
  'Pyo',
  'Chi',
  'Su',
  '☺️',
  'probably',
  'talk',
  'individual',
  'characters',
  'next',
  'time',
  'CLOY',
  'tackled',
  'politics',
  'drama',
  'deception',
  'family',
  'issues',
  'friendships',
  'best',
  'pure',
  'unconditional',
  'love',
  'love',
  'brought',
  'fate',
  'one',
  'highly',
  'entertaining',
  'well-written',
  'package',
  '’'

In [22]:
# every unique word becomes one dimension
vocab = sorted(set(w for words in newWordlist for w in words))
dfNew = pd.DataFrame(
    [Counter(words) for words in newWordlist]
).reindex(columns=vocab).fillna(0).astype(int)

dfNew.index = [f"d{i+1}" for i in range(len(dfNew))]

In [23]:
dfNew.head

<bound method NDFrame.head of        '-i  '-sshi  'D  'LL  'M  'RE  'S  'd  'll  'm  ...  🫰🫰🫰🫰  🫶  🫶🍱  🫶🏻  \
d1       0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d2       0       0   0    0   0    0   0   1    2   7  ...     0  0   0   0   
d3       0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d4       0       0   0    0   0    0   0   1    0   1  ...     0  0   0   0   
d5       0       0   0    0   0    0   0   2    1   3  ...     0  0   0   0   
...    ...     ...  ..  ...  ..  ...  ..  ..  ...  ..  ...   ... ..  ..  ..   
d3071    0       0   0    0   0    0   0   0    1   0  ...     0  0   0   0   
d3072    0       0   0    0   0    0   0   0    0   1  ...     0  0   0   0   
d3073    0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d3074    0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   
d3075    0       0   0    0   0    0   0   0    0   0  ...     0  0   0   0   

       🫶🏻😂😂  🫶🏻🫶🏻🫶🏻  

In [24]:
M_new = dfNew.values

scores_new = []
for i, j in combinations(range(len(dfNew)), 2):
    score = cosineSimilarity(M_new[i], M_new[j])
    scores_new.append({
        "thread 1": loaded_cloy[i]["post_id"],
        "thread 2": loaded_cloy[j]["post_id"],
        "similarity score": score
    })

In [25]:
scores_new_df = pd.DataFrame(scores_new)
scores_new_df

,thread 1,thread 2,similarity score
0,dkxewd,ffh52z,0.000000
1,dkxewd,fguorr,0.000000
2,dkxewd,fhcrpl,0.000000
3,dkxewd,fj5byj,0.000000
4,dkxewd,fjimic,0.000000
...,...,...,...
4726270,1s7yg7f,1s8i9nn,0.207840
4726271,1s7yg7f,1s8ju38,0.129580
4726272,1s8aeiv,1s8i9nn,0.047782
4726273,1s8aeiv,1s8ju38,0.023328


In [26]:
scores_new_df_sorted = scores_new_df.sort_values(by="similarity score", ascending=False)

def getSimilarNew(threadID):
    sim = scores_new_df_sorted[scores_new_df_sorted["thread 1"] == threadID][:5]
    title = titles[threadID]
    top5 = []
    for index, row in sim.iterrows():
        top5.append(titles[row["thread 2"]])
    print("Thread ID: ", threadID)
    print("Thread Title: ", title)
    print("Top 5 Subject Titles: ", top5)


In [27]:
getSimilarNew("1hm20kg")

Thread ID:  1hm20kg
Thread Title:  This scene really hits me hard.
Top 5 Subject Titles:  ['This scene really hits me hard.', "what's your favorite scene/moment in crash landing on you?", 'Do you remember these scenes', 'This scene is truly heart breaking , even if it is 5 years old. Both are fiercely protective of each and would sacrifice themselves to do so. Spellbinding and heart wrenching 💔💔💔💔💔', 'Underrated moment #7: "Se-ri you just woke up. You can\'t take off his shirt like that already!"']


In [28]:
getSimilar("1hm20kg")

Thread ID:  1hm20kg
Thread Title:  This scene really hits me hard.
Top 5 Subject Titles:  ['This scene really hits me hard.', "Both hurting, both won't let go. ", 'Underrated moment #7: "Se-ri you just woke up. You can\'t take off his shirt like that already!"', "It's Hyun Bin's birthday! Share your favourite moments of HB from CLOY! Gifs especially welcome 🤗", "I've watched this show over a dozen times but Seo Dan and Gu Seung Jun really grew on me"]


In [29]:
scores_df_sorted.describe()

,similarity score
count,4.726275e+06
mean,3.915300e-01
std,2.344973e-01
min,0.000000e+00
25%,1.929253e-01
50%,4.082580e-01
75%,5.813526e-01
max,1.000000e+00


In [30]:
scores_new_df_sorted.describe()

,similarity score
count,4.726275e+06
mean,1.365073e-01
std,1.230201e-01
min,0.000000e+00
25%,3.571494e-02
50%,1.096568e-01
75%,2.069112e-01
max,1.000000e+00


By comparing the profiles of the similarity scores before and after removing stopwords, we can see that stopwords did affect our results. 
They made comments more similar to each others. 
The Top 5 threads that are similar to "1hm20kg" also differ. 
